In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
%pip install -q psxdata pandas numpy plotly ipywidgets

In [ ]:
import warnings
import json
import time
from datetime import datetime, date, timedelta
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from IPython.display import display, HTML

warnings.filterwarnings("ignore")

# Dashboard branding
PROJECT_NAME = "ZENSTAR024"
DASHBOARD_NAME = "PSX MARKET ANALYTICS"
CURRENCY = "PKR"

# Change these settings when needed
DEFAULT_SYMBOL = "OGDC"
HISTORY_YEARS = 2
MAX_COMPARE_STOCKS = 5

# Local output directory
OUTPUT_DIR = Path("/kaggle/working/zenstar024_psx")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Professional dark trading-terminal theme
BG = "#080D17"
PANEL = "#111827"
TEXT = "#E5EDF8"
MUTED = "#93A4BA"
GREEN = "#20D6A1"
RED = "#FF5D73"
GOLD = "#F3BD65"
BLUE = "#66A9FF"

print(f"{PROJECT_NAME} | {DASHBOARD_NAME}")
print("Currency:", CURRENCY)
print("Output directory:", OUTPUT_DIR)
print("Notebook date:", datetime.now().astimezone().isoformat())

In [ ]:
try:
    import psxdata
    print("psxdata imported successfully.")
except Exception as exc:
    raise RuntimeError(
        "Unable to import psxdata. Run Cell 1 again, "
        "check Kaggle Internet settings, and restart the kernel if needed."
    ) from exc


def as_dataframe(value):
    """Convert supported data results to a DataFrame."""
    if isinstance(value, pd.DataFrame):
        return value.copy()

    if isinstance(value, dict):
        if isinstance(value.get("data"), list):
            return pd.DataFrame(value["data"])
        if isinstance(value.get("results"), list):
            return pd.DataFrame(value["results"])
        try:
            return pd.DataFrame(value)
        except Exception:
            return pd.DataFrame()

    if isinstance(value, (list, tuple)):
        return pd.DataFrame(value)

    return pd.DataFrame()


def find_column(df, names):
    """Find a column without depending on capitalization."""
    lookup = {str(c).strip().lower(): c for c in df.columns}
    for name in names:
        if name.lower() in lookup:
            return lookup[name.lower()]
    return None


# Retrieve the available symbols
symbols = []
screener_df = pd.DataFrame()
symbol_error = None
screener_error = None

try:
    raw_symbols = psxdata.tickers()
    if isinstance(raw_symbols, pd.DataFrame):
        symbol_col = find_column(
            raw_symbols, ["symbol", "ticker", "code", "scrip"]
        )
        if symbol_col:
            symbols = raw_symbols[symbol_col].dropna().astype(str).tolist()
        else:
            symbols = raw_symbols.index.astype(str).tolist()
    elif isinstance(raw_symbols, (list, tuple, set, np.ndarray)):
        symbols = [str(x) for x in raw_symbols if pd.notna(x)]
    else:
        symbols = []
except Exception as exc:
    symbol_error = str(exc)

try:
    screener_df = as_dataframe(psxdata.screener())
except Exception as exc:
    screener_error = str(exc)

# If the screener supplies symbols, use them to supplement the universe.
if not screener_df.empty:
    sc = find_column(screener_df, ["symbol", "ticker", "code", "scrip"])
    if sc:
        symbols.extend(screener_df[sc].dropna().astype(str).tolist())

symbols = sorted({
    s.strip().upper()
    for s in symbols
    if s and s.strip() and s.lower() != "nan"
})

print("Symbols retrieved:", len(symbols))
print("Screener rows retrieved:", len(screener_df))

if symbols:
    display(pd.DataFrame({"Available PSX symbol": symbols}).head(20))
else:
    print("No symbol list was returned by the source.")

if symbol_error:
    print("Symbol retrieval error:", symbol_error)

if screener_error:
    print("Screener retrieval error:", screener_error)

if not symbols:
    raise RuntimeError(
        "No stock symbols were retrieved. Check the library documentation, "
        "Kaggle Internet access, and the data source status before continuing."
    )

In [ ]:

# CELL 4 — HISTORICAL PRICES AND LATEST AVAILABLE QUOTE

from datetime import date, timedelta
import warnings
import pandas as pd
import numpy as np
import psxdata


def as_dataframe(value):
    """Convert source results to a DataFrame safely."""
    if isinstance(value, pd.DataFrame):
        return value.copy()

    if isinstance(value, dict):
        if isinstance(value.get("data"), list):
            return pd.DataFrame(value["data"])
        if isinstance(value.get("results"), list):
            return pd.DataFrame(value["results"])
        try:
            return pd.DataFrame([value])
        except Exception:
            return pd.DataFrame()

    if isinstance(value, (list, tuple)):
        return pd.DataFrame(value)

    return pd.DataFrame()


def find_column(df, names):
    """Find columns regardless of capitalization."""
    lookup = {
        str(col).strip().lower(): col
        for col in df.columns
    }

    for name in names:
        if name.lower() in lookup:
            return lookup[name.lower()]

    return None


def normalize_ohlcv(raw):
    """Normalize, validate, and sort historical OHLCV data."""
    df = as_dataframe(raw)

    if df.empty:
        return pd.DataFrame(
            columns=[
                "date", "open", "high", "low",
                "close", "volume"
            ]
        )

    # Convert a date index to a regular column if needed.
    date_names = [
        "date", "datetime", "timestamp", "time"
    ]

    if not any(
        str(col).strip().lower() in date_names
        for col in df.columns
    ):
        df = df.reset_index()

    aliases = {
        "date": [
            "date", "datetime", "timestamp", "time", "index"
        ],
        "open": ["open", "opening", "open_price"],
        "high": ["high", "high_price"],
        "low": ["low", "low_price"],
        "close": [
            "close", "closing", "close_price",
            "adj close", "adj_close"
        ],
        "volume": [
            "volume", "vol", "traded_volume", "shares_traded"
        ],
    }

    rename = {}

    for target, choices in aliases.items():
        col = find_column(df, choices)
        if col is not None:
            rename[col] = target

    df = df.rename(columns=rename)

    required = ["date", "open", "high", "low", "close"]
    missing = [col for col in required if col not in df.columns]

    if missing:
        raise ValueError(
            f"Missing required columns: {missing}. "
            f"Source returned: {list(df.columns)}"
        )

    if "volume" not in df.columns:
        df["volume"] = np.nan

    # Parse dates, including timezone-aware timestamps.
    parsed_dates = pd.to_datetime(
        df["date"], errors="coerce", utc=True
    )

    df["date"] = parsed_dates.dt.tz_convert(None)

    for col in ["open", "high", "low", "close", "volume"]:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    # Remove invalid dates and missing OHLC prices.
    df = df.dropna(
        subset=["date", "open", "high", "low", "close"]
    ).copy()

    # Reject impossible/non-positive prices.
    df = df[
        (df[["open", "high", "low", "close"]] > 0).all(axis=1)
        & (df["high"] >= df["low"])
        & (df["high"] >= df[["open", "close"]].max(axis=1))
        & (df["low"] <= df[["open", "close"]].min(axis=1))
    ].copy()

    # Sort oldest to newest, then remove duplicate dates.
    df = df.sort_values("date", ascending=True)
    df = df.drop_duplicates(subset=["date"], keep="last")
    df = df.reset_index(drop=True)

    return df[
        ["date", "open", "high", "low", "close", "volume"]
    ]


def get_history(symbol, years=2):
    """Download historical prices and return ascending dates."""
    end_date = date.today()
    start_date = end_date - timedelta(days=365 * years)

    # This warning can be produced by the source before we normalize
    # and sort its returned records. We still validate the final data.
    with warnings.catch_warnings():
        warnings.filterwarnings(
            "ignore",
            message=".*Dates are not in chronological order.*"
        )

        raw = psxdata.stocks(
            symbol,
            start=start_date.isoformat(),
            end=end_date.isoformat()
        )

    df = normalize_ohlcv(raw)

    if df.empty:
        raise ValueError(
            f"No valid historical prices returned for {symbol}."
        )

    df["symbol"] = symbol

    # Final chronological-order verification.
    if not df["date"].is_monotonic_increasing:
        raise ValueError(
            f"Historical dates are still not ordered for {symbol}."
        )

    return df


def get_quote(symbol):
    """Retrieve a separate quote if supported by the data source."""
    try:
        result = psxdata.quote(symbol)
        df = as_dataframe(result)

        if df.empty:
            return pd.DataFrame()

        return df

    except Exception as exc:
        print(f"Separate quote unavailable for {symbol}: {exc}")
        return pd.DataFrame()


# Select the main stock.
if not symbols:
    raise RuntimeError(
        "No PSX symbols are available. Run Cell 3 successfully first."
    )

selected_symbol = (
    DEFAULT_SYMBOL
    if DEFAULT_SYMBOL in symbols
    else symbols[0]
)

print("Selected symbol:", selected_symbol)

try:
    history = get_history(selected_symbol, HISTORY_YEARS)

    # Recalculate indicators using the corrected date order.
    history = add_indicators(history)

    print("Historical rows:", len(history))
    print("First date:", history["date"].min())
    print("Last date:", history["date"].max())
    print(
        "Dates sorted correctly:",
        history["date"].is_monotonic_increasing
    )

    display(history.tail(10).round(3))

except Exception as exc:
    history = pd.DataFrame()
    print("Historical download failed:", exc)

latest_quote = get_quote(selected_symbol)

if not latest_quote.empty:
    print("Separate quote returned by the source:")
    display(latest_quote.head())
else:
    print(
        "No separate quote returned. "
        "Historical data is kept separate from live quotes."
    )


In [ ]:
def add_indicators(df):
    """Calculate indicators using historical prices only."""
    out = df.copy().sort_values("date")

    close = out["close"]
    out["daily_return_pct"] = close.pct_change() * 100
    out["SMA_20"] = close.rolling(20, min_periods=20).mean()
    out["SMA_50"] = close.rolling(50, min_periods=50).mean()
    out["EMA_20"] = close.ewm(span=20, adjust=False).mean()

    delta = close.diff()
    gain = delta.clip(lower=0)
    loss = -delta.clip(upper=0)

    avg_gain = gain.ewm(alpha=1/14, min_periods=14, adjust=False).mean()
    avg_loss = loss.ewm(alpha=1/14, min_periods=14, adjust=False).mean()

    rs = avg_gain / avg_loss.replace(0, np.nan)
    out["RSI_14"] = 100 - (100 / (1 + rs))

    # MACD
    ema12 = close.ewm(span=12, adjust=False).mean()
    ema26 = close.ewm(span=26, adjust=False).mean()
    out["MACD"] = ema12 - ema26
    out["MACD_signal"] = out["MACD"].ewm(span=9, adjust=False).mean()
    out["MACD_hist"] = out["MACD"] - out["MACD_signal"]

    # Annualized historical volatility using daily returns.
    out["volatility_20d_pct"] = (
        out["daily_return_pct"].rolling(20).std() * np.sqrt(252)
    )

    out["rolling_high_52w"] = close.rolling(252, min_periods=1).max()
    out["rolling_low_52w"] = close.rolling(252, min_periods=1).min()

    return out


if not history.empty:
    history = add_indicators(history)

    display(
        history[
            [
                "date", "open", "high", "low", "close", "volume",
                "daily_return_pct", "SMA_20", "SMA_50",
                "RSI_14", "MACD"
            ]
        ].tail(10).round(3)
    )
else:
    print("Indicators cannot be calculated until historical prices load.")

In [ ]:
def plot_stock_chart(df, symbol):
    if df.empty:
        print(f"No historical data available to chart for {symbol}.")
        return

    d = df.copy().sort_values("date")

    fig = make_subplots(
        rows=3, cols=1,
        shared_xaxes=True,
        vertical_spacing=0.035,
        row_heights=[0.58, 0.20, 0.22],
        subplot_titles=(
            f"{symbol} — OHLC Candlestick",
            "Trading Volume",
            "RSI (14)"
        )
    )

    fig.add_trace(
        go.Candlestick(
            x=d["date"],
            open=d["open"],
            high=d["high"],
            low=d["low"],
            close=d["close"],
            name="OHLC",
            increasing_line_color=GREEN,
            decreasing_line_color=RED
        ),
        row=1, col=1
    )

    for column, color in [
        ("SMA_20", GOLD),
        ("SMA_50", BLUE),
        ("EMA_20", "#C084FC")
    ]:
        if column in d.columns:
            fig.add_trace(
                go.Scatter(
                    x=d["date"],
                    y=d[column],
                    name=column,
                    line=dict(color=color, width=1.5)
                ),
                row=1, col=1
            )

    volume_colors = [
        GREEN if c >= o else RED
        for o, c in zip(d["open"], d["close"])
    ]

    fig.add_trace(
        go.Bar(
            x=d["date"],
            y=d["volume"],
            name="Volume",
            marker_color=volume_colors
        ),
        row=2, col=1
    )

    fig.add_trace(
        go.Scatter(
            x=d["date"],
            y=d["RSI_14"],
            name="RSI 14",
            line=dict(color=GOLD, width=1.6)
        ),
        row=3, col=1
    )

    fig.add_hline(
        y=70, line_dash="dash", line_color=RED,
        opacity=0.65, row=3, col=1
    )
    fig.add_hline(
        y=30, line_dash="dash", line_color=GREEN,
        opacity=0.65, row=3, col=1
    )

    fig.update_layout(
        template="plotly_dark",
        paper_bgcolor=BG,
        plot_bgcolor=PANEL,
        font=dict(color=TEXT),
        height=850,
        title=dict(
            text=f"{PROJECT_NAME}  /  {symbol}  /  DAILY ANALYSIS",
            x=0.02
        ),
        xaxis_rangeslider_visible=False,
        legend=dict(orientation="h", y=1.02, x=0),
        margin=dict(l=35, r=25, t=95, b=35),
        hovermode="x unified"
    )

    fig.update_yaxes(gridcolor="#263247", zerolinecolor="#263247")
    fig.update_xaxes(gridcolor="#263247")
    fig.update_yaxes(title_text="Price (PKR)", row=1, col=1)
    fig.update_yaxes(title_text="Shares", row=2, col=1)
    fig.update_yaxes(title_text="RSI", range=[0, 100], row=3, col=1)

    fig.show()


if not history.empty:
    plot_stock_chart(history, selected_symbol)

In [ ]:
def format_pkr(value):
    if pd.isna(value):
        return "N/A"
    return f"PKR {value:,.2f}"


def show_stock_summary(df, symbol):
    if df.empty:
        print("Summary unavailable: no valid historical prices.")
        return

    last = df.iloc[-1]
    previous = df.iloc[-2] if len(df) > 1 else None

    close = last["close"]
    daily_change = (
        (close / previous["close"] - 1) * 100
        if previous is not None and previous["close"] != 0
        else np.nan
    )

    first_close = df.iloc[0]["close"]
    period_return = (
        (close / first_close - 1) * 100
        if first_close != 0 else np.nan
    )

    high = df["high"].max()
    low = df["low"].min()
    avg_volume = df["volume"].mean()

    change_color = GREEN if pd.notna(daily_change) and daily_change >= 0 else RED
    change_text = (
        f"{daily_change:+.2f}%"
        if pd.notna(daily_change) else "N/A"
    )

    display(HTML(f"""
    <div style="background:linear-gradient(130deg,#111827,#182338);
                border:1px solid #34445c;border-radius:18px;
                padding:24px;color:#e5edf8;font-family:Arial,sans-serif;">
      <div style="color:#93a4ba;font-size:12px;letter-spacing:2px;">
        ZENSTAR024 · HISTORICAL MARKET SUMMARY
      </div>
      <h2 style="margin:8px 0;">{symbol}</h2>
      <div style="font-size:32px;font-weight:700;">
        {format_pkr(close)}
      </div>
      <div style="color:{change_color};font-weight:700;margin-top:6px;">
        Daily close-to-close return: {change_text}
      </div>
      <hr style="border-color:#34445c;">
      <div style="display:grid;grid-template-columns:repeat(3,minmax(0,1fr));
                  gap:18px;">
        <div><small>Period return</small><h3>{period_return:+.2f}%</h3></div>
        <div><small>Highest observed high</small><h3>{format_pkr(high)}</h3></div>
        <div><small>Lowest observed low</small><h3>{format_pkr(low)}</h3></div>
        <div><small>Latest candle date</small><h3>{last['date'].date()}</h3></div>
        <div><small>Average daily volume</small><h3>{avg_volume:,.0f}</h3></div>
        <div><small>Valid observations</small><h3>{len(df):,}</h3></div>
      </div>
    </div>
    """))


if not history.empty:
    show_stock_summary(history, selected_symbol)

In [ ]:

# CELL 4 — HISTORICAL PRICES AND LATEST AVAILABLE QUOTE

from datetime import date, timedelta
import warnings
import pandas as pd
import numpy as np
import psxdata


def as_dataframe(value):
    """Convert source results to a DataFrame safely."""
    if isinstance(value, pd.DataFrame):
        return value.copy()

    if isinstance(value, dict):
        if isinstance(value.get("data"), list):
            return pd.DataFrame(value["data"])
        if isinstance(value.get("results"), list):
            return pd.DataFrame(value["results"])
        try:
            return pd.DataFrame([value])
        except Exception:
            return pd.DataFrame()

    if isinstance(value, (list, tuple)):
        return pd.DataFrame(value)

    return pd.DataFrame()


def find_column(df, names):
    """Find columns regardless of capitalization."""
    lookup = {
        str(col).strip().lower(): col
        for col in df.columns
    }

    for name in names:
        if name.lower() in lookup:
            return lookup[name.lower()]

    return None


def normalize_ohlcv(raw):
    """Normalize, validate, and sort historical OHLCV data."""
    df = as_dataframe(raw)

    if df.empty:
        return pd.DataFrame(
            columns=[
                "date", "open", "high", "low",
                "close", "volume"
            ]
        )

    # Convert a date index to a regular column if needed.
    date_names = [
        "date", "datetime", "timestamp", "time"
    ]

    if not any(
        str(col).strip().lower() in date_names
        for col in df.columns
    ):
        df = df.reset_index()

    aliases = {
        "date": [
            "date", "datetime", "timestamp", "time", "index"
        ],
        "open": ["open", "opening", "open_price"],
        "high": ["high", "high_price"],
        "low": ["low", "low_price"],
        "close": [
            "close", "closing", "close_price",
            "adj close", "adj_close"
        ],
        "volume": [
            "volume", "vol", "traded_volume", "shares_traded"
        ],
    }

    rename = {}

    for target, choices in aliases.items():
        col = find_column(df, choices)
        if col is not None:
            rename[col] = target

    df = df.rename(columns=rename)

    required = ["date", "open", "high", "low", "close"]
    missing = [col for col in required if col not in df.columns]

    if missing:
        raise ValueError(
            f"Missing required columns: {missing}. "
            f"Source returned: {list(df.columns)}"
        )

    if "volume" not in df.columns:
        df["volume"] = np.nan

    # Parse dates, including timezone-aware timestamps.
    parsed_dates = pd.to_datetime(
        df["date"], errors="coerce", utc=True
    )

    df["date"] = parsed_dates.dt.tz_convert(None)

    for col in ["open", "high", "low", "close", "volume"]:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    # Remove invalid dates and missing OHLC prices.
    df = df.dropna(
        subset=["date", "open", "high", "low", "close"]
    ).copy()

    # Reject impossible/non-positive prices.
    df = df[
        (df[["open", "high", "low", "close"]] > 0).all(axis=1)
        & (df["high"] >= df["low"])
        & (df["high"] >= df[["open", "close"]].max(axis=1))
        & (df["low"] <= df[["open", "close"]].min(axis=1))
    ].copy()

    # Sort oldest to newest, then remove duplicate dates.
    df = df.sort_values("date", ascending=True)
    df = df.drop_duplicates(subset=["date"], keep="last")
    df = df.reset_index(drop=True)

    return df[
        ["date", "open", "high", "low", "close", "volume"]
    ]


def get_history(symbol, years=2):
    """Download historical prices and return ascending dates."""
    end_date = date.today()
    start_date = end_date - timedelta(days=365 * years)

    # This warning can be produced by the source before we normalize
    # and sort its returned records. We still validate the final data.
    with warnings.catch_warnings():
        warnings.filterwarnings(
            "ignore",
            message=".*Dates are not in chronological order.*"
        )

        raw = psxdata.stocks(
            symbol,
            start=start_date.isoformat(),
            end=end_date.isoformat()
        )

    df = normalize_ohlcv(raw)

    if df.empty:
        raise ValueError(
            f"No valid historical prices returned for {symbol}."
        )

    df["symbol"] = symbol

    # Final chronological-order verification.
    if not df["date"].is_monotonic_increasing:
        raise ValueError(
            f"Historical dates are still not ordered for {symbol}."
        )

    return df


def get_quote(symbol):
    """Retrieve a separate quote if supported by the data source."""
    try:
        result = psxdata.quote(symbol)
        df = as_dataframe(result)

        if df.empty:
            return pd.DataFrame()

        return df

    except Exception as exc:
        print(f"Separate quote unavailable for {symbol}: {exc}")
        return pd.DataFrame()


# Select the main stock.
if not symbols:
    raise RuntimeError(
        "No PSX symbols are available. Run Cell 3 successfully first."
    )

selected_symbol = (
    DEFAULT_SYMBOL
    if DEFAULT_SYMBOL in symbols
    else symbols[0]
)

print("Selected symbol:", selected_symbol)

try:
    history = get_history(selected_symbol, HISTORY_YEARS)

    # Recalculate indicators using the corrected date order.
    history = add_indicators(history)

    print("Historical rows:", len(history))
    print("First date:", history["date"].min())
    print("Last date:", history["date"].max())
    print(
        "Dates sorted correctly:",
        history["date"].is_monotonic_increasing
    )

    display(history.tail(10).round(3))

except Exception as exc:
    history = pd.DataFrame()
    print("Historical download failed:", exc)

latest_quote = get_quote(selected_symbol)

if not latest_quote.empty:
    print("Separate quote returned by the source:")
    display(latest_quote.head())
else:
    print(
        "No separate quote returned. "
        "Historical data is kept separate from live quotes."
    )


In [ ]:
print("Number of available symbols:", len(symbols))
print("Example symbols:", symbols[:60])

compare_input = input(
    "Enter up to 5 symbols separated by commas (e.g. OGDC,HBL,MCB): "
).strip()

compare_symbols = [
    s.strip().upper()
    for s in compare_input.split(",")
    if s.strip()
][:MAX_COMPARE_STOCKS]

compare_symbols = [s for s in compare_symbols if s in symbols]

if not compare_symbols:
    compare_symbols = [selected_symbol]
    print("No valid comparison input; using the selected symbol.")

comparison_prices = {}
comparison_errors = {}

for symbol in compare_symbols:
    try:
        df = get_history(symbol, HISTORY_YEARS)
        if not df.empty:
            comparison_prices[symbol] = df.set_index("date")["close"]
        else:
            comparison_errors[symbol] = "No valid historical observations"
    except Exception as exc:
        comparison_errors[symbol] = str(exc)

if comparison_prices:
    prices = pd.DataFrame(comparison_prices).sort_index()
    normalized_returns = prices.divide(prices.iloc[0]).multiply(100)

    fig = go.Figure()

    for symbol in normalized_returns.columns:
        fig.add_trace(
            go.Scatter(
                x=normalized_returns.index,
                y=normalized_returns[symbol],
                name=symbol,
                mode="lines"
            )
        )

    fig.update_layout(
        title="PSX Company Comparison — Normalized Starting Value = 100",
        template="plotly_dark",
        paper_bgcolor=BG,
        plot_bgcolor=PANEL,
        font=dict(color=TEXT),
        yaxis_title="Indexed closing price",
        xaxis_title="Date",
        hovermode="x unified",
        height=550
    )
    fig.show()

    comparison_summary = pd.DataFrame({
        "First close": prices.iloc[0],
        "Latest close": prices.ffill().iloc[-1],
        "Period return (%)": (
            prices.ffill().iloc[-1] / prices.iloc[0] - 1
        ) * 100
    })

    display(comparison_summary.round(3))
    comparison_summary.to_csv(
        OUTPUT_DIR / "psx_company_comparison.csv"
    )
else:
    print("No comparison data could be retrieved.")

if comparison_errors:
    print("Comparison errors:")
    display(pd.DataFrame(
        list(comparison_errors.items()),
        columns=["Symbol", "Error"]
    ))

In [ ]:
watchlist_input = input(
    "Enter watchlist symbols separated by commas (e.g. OGDC,HBL,LUCK): "
).strip()

watchlist = list(dict.fromkeys(
    s.strip().upper()
    for s in watchlist_input.split(",")
    if s.strip() and s.strip().upper() in symbols
))

watchlist_records = []

for symbol in watchlist:
    try:
        df = get_history(symbol, years=1)

        if df.empty:
            continue

        latest = df.iloc[-1]
        previous_close = (
            df.iloc[-2]["close"] if len(df) > 1 else np.nan
        )

        daily_change = (
            (latest["close"] / previous_close - 1) * 100
            if pd.notna(previous_close) and previous_close != 0
            else np.nan
        )

        watchlist_records.append({
            "Symbol": symbol,
            "Latest available close (PKR)": latest["close"],
            "Previous close (PKR)": previous_close,
            "Daily return (%)": daily_change,
            "Volume": latest["volume"],
            "Last historical date": latest["date"]
        })

    except Exception as exc:
        print(f"{symbol}: retrieval failed — {exc}")

watchlist_df = pd.DataFrame(watchlist_records)

if not watchlist_df.empty:
    display(watchlist_df.round(3))
    watchlist_df.to_csv(OUTPUT_DIR / "psx_watchlist.csv", index=False)
else:
    print("No watchlist data available. Check your symbols and connection.")

In [ ]:
print("Portfolio input format:")
print("SYMBOL:BUY_PRICE:QUANTITY,SYMBOL:BUY_PRICE:QUANTITY")
print("Example format only: OGDC:250:10,HBL:150:20")
print("Replace the example values with your actual purchase details.")

portfolio_input = input("Enter your portfolio positions: ").strip()

portfolio_records = []
portfolio_errors = []

if portfolio_input:
    for item in portfolio_input.split(","):
        parts = item.strip().split(":")

        if len(parts) != 3:
            portfolio_errors.append(
                f"Invalid position format: {item}"
            )
            continue

        symbol = parts[0].strip().upper()

        try:
            buy_price = float(parts[1])
            quantity = float(parts[2])

            if buy_price <= 0 or quantity <= 0:
                raise ValueError("Price and quantity must be positive.")

            if symbol not in symbols:
                portfolio_errors.append(
                    f"{symbol}: not found in the retrieved stock universe."
                )
                continue

            df = get_history(symbol, years=1)

            if df.empty:
                portfolio_errors.append(
                    f"{symbol}: no valid historical close available."
                )
                continue

            latest = df.iloc[-1]
            current_price = float(latest["close"])
            cost_basis = buy_price * quantity
            market_value = current_price * quantity
            profit_loss = market_value - cost_basis
            return_pct = (profit_loss / cost_basis) * 100

            portfolio_records.append({
                "Symbol": symbol,
                "Buy price (PKR)": buy_price,
                "Quantity": quantity,
                "Latest available close (PKR)": current_price,
                "Purchase cost (PKR)": cost_basis,
                "Market value at last close (PKR)": market_value,
                "Unrealized P/L (PKR)": profit_loss,
                "Return (%)": return_pct,
                "Price date": latest["date"]
            })

        except Exception as exc:
            portfolio_errors.append(f"{symbol}: {exc}")

portfolio_df = pd.DataFrame(portfolio_records)

if not portfolio_df.empty:
    display(portfolio_df.round(2))

    total_cost = portfolio_df["Purchase cost (PKR)"].sum()
    total_value = portfolio_df["Market value at last close (PKR)"].sum()
    total_pl = portfolio_df["Unrealized P/L (PKR)"].sum()
    total_return = (
        total_pl / total_cost * 100 if total_cost else np.nan
    )

    print(f"Total purchase cost: {format_pkr(total_cost)}")
    print(f"Portfolio value at available closes: {format_pkr(total_value)}")
    print(f"Unrealized profit/loss: {format_pkr(total_pl)}")
    print(f"Weighted portfolio return: {total_return:.2f}%")

    portfolio_df.to_csv(
        OUTPUT_DIR / "psx_portfolio_analysis.csv", index=False
    )
else:
    print("No valid portfolio positions were calculated.")

if portfolio_errors:
    print("\nPositions that could not be calculated:")
    for error in portfolio_errors:
        print("-", error)

In [ ]:
if not history.empty:
    historical_export = history.copy()
    historical_export["source_symbol"] = selected_symbol

    history_file = OUTPUT_DIR / f"{selected_symbol}_historical_analysis.csv"
    historical_export.to_csv(history_file, index=False)

    print("Historical analysis saved:", history_file)
    print("Available columns:", list(historical_export.columns))

    display(
        historical_export.tail(10).round(3)
    )
else:
    print("No historical data to export.")

In [ ]:
print("=" * 65)
print(f"{PROJECT_NAME} | DATA QUALITY REPORT")
print("=" * 65)

print("Report generated:", datetime.now().astimezone().isoformat())
print("Data source library:", "psxdata")
print("Selected stock:", selected_symbol)

if not history.empty:
    print("Historical observations:", len(history))
    print("First historical candle:", history["date"].min())
    print("Last historical candle:", history["date"].max())
    print("Latest close:", format_pkr(history.iloc[-1]["close"]))
    print("Historical data retrieved during this notebook run.")
else:
    print("Historical data status: unavailable")

if not latest_quote.empty:
    print("Separate quote response: received")
    print("Quote columns:", list(latest_quote.columns))
else:
    print("Separate quote response: unavailable")

print("Screener records:", len(screener_df))
print("Symbols retrieved:", len(symbols))

print("\nData interpretation:")
print("- A historical close is not necessarily a live quote.")
print("- A missing price is not replaced with a fabricated price.")
print("- Technical indicators are calculated from retrieved history.")
print("- Source timestamps should be used when the provider supplies them.")
print("- Company fundamentals and market capitalization may be unavailable.")
print("- A successful download does not guarantee that data is real-time.")

In [ ]:
print("Files created during this run:")

for path in sorted(OUTPUT_DIR.glob("*.csv")):
    print(f"- {path.name} ({path.stat().st_size:,} bytes)")

# Bundle generated CSV files into one ZIP for convenient download.
import zipfile

zip_path = OUTPUT_DIR / "zenstar024_psx_exports.zip"

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as archive:
    for path in OUTPUT_DIR.glob("*.csv"):
        archive.write(path, arcname=path.name)

print("\nZIP export created:", zip_path)

In [ ]:

# CELL 15 — CREATE A REAL PSX HISTORICAL CSV DATASET

import psxdata
import pandas as pd
from datetime import date, timedelta
from pathlib import Path
from IPython.display import display

DATA_DIR = Path("/kaggle/working/zenstar024_psx")
DATA_DIR.mkdir(parents=True, exist_ok=True)

# Start with a small batch to avoid excessive requests.
# These are requested symbols, not guaranteed to all be available.
START_DATE = date.today() - timedelta(days=365 * 3)
END_DATE = date.today()

try:
    available_tickers = psxdata.tickers()

    if isinstance(available_tickers, pd.DataFrame):
        ticker_col = next(
            (
                c for c in available_tickers.columns
                if str(c).lower() in ["symbol", "ticker", "code"]
            ),
            None
        )
        ticker_list = (
            available_tickers[ticker_col].dropna().astype(str).tolist()
            if ticker_col else available_tickers.index.astype(str).tolist()
        )
    else:
        ticker_list = [str(s) for s in available_tickers]

    ticker_list = sorted(set(s.strip().upper() for s in ticker_list))

    print("Available symbols retrieved:", len(ticker_list))

except Exception as exc:
    raise RuntimeError(
        f"Could not retrieve PSX symbols from the data source: {exc}"
    )

# Select up to 10 symbols for the first download.
preferred = ["OGDC", "HBL", "MCB", "LUCK", "PPL",
             "PSO", "FFC", "ENGRO", "SYS", "UBL"]

batch = [s for s in preferred if s in ticker_list]
batch += [s for s in ticker_list if s not in batch][:max(0, 10 - len(batch))]

frames = []
failed = []

for symbol in batch:
    try:
        df = psxdata.stocks(
            symbol,
            start=START_DATE.isoformat(),
            end=END_DATE.isoformat()
        )

        if df is None or df.empty:
            failed.append((symbol, "No data returned"))
            continue

        df = df.copy()
        df.columns = [str(c).strip().lower() for c in df.columns]

        if "date" not in df.columns:
            failed.append((symbol, "Missing date column"))
            continue

        required = ["open", "high", "low", "close"]
        if not all(c in df.columns for c in required):
            failed.append((symbol, "Missing OHLC columns"))
            continue

        if "volume" not in df.columns:
            df["volume"] = pd.NA

        df["date"] = pd.to_datetime(df["date"], errors="coerce", utc=True)
        df["date"] = df["date"].dt.tz_convert(None)

        for col in required + ["volume"]:
            df[col] = pd.to_numeric(df[col], errors="coerce")

        df = df.dropna(subset=["date"] + required)
        df = df.sort_values("date").drop_duplicates("date")
        df["symbol"] = symbol

        if not df.empty:
            frames.append(
                df[["symbol", "date", "open", "high", "low", "close", "volume"]]
            )
            print(f"{symbol}: {len(df)} historical rows saved")
        else:
            failed.append((symbol, "No valid OHLC rows"))

    except Exception as exc:
        failed.append((symbol, str(exc)))
        print(f"{symbol}: download failed — {exc}")

if not frames:
    raise RuntimeError(
        "No historical data could be downloaded. "
        "Check the PSX connection and retry."
    )

psx_dataset = pd.concat(frames, ignore_index=True)
psx_dataset = psx_dataset.sort_values(["symbol", "date"]).reset_index(drop=True)

csv_path = DATA_DIR / "psx_historical_dataset.csv"
psx_dataset.to_csv(csv_path, index=False)

print("\nDataset created successfully.")
print("Rows:", len(psx_dataset))
print("Companies:", psx_dataset["symbol"].nunique())
print("First date:", psx_dataset["date"].min())
print("Last date:", psx_dataset["date"].max())
print("CSV path:", csv_path)

display(psx_dataset.head(10))

if failed:
    print("\nSymbols not downloaded:")
    display(pd.DataFrame(failed, columns=["Symbol", "Reason"]))
